## 准备数据

In [6]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [7]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [8]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        # 输入层: 28*28 = 784
        # 隐藏层: 100个神经元
        # 输出层: 10个类别
        self.W1 = tf.Variable(shape=[28*28, 100], dtype=tf.float32, 
                             initial_value=tf.random.normal([28*28, 100], stddev=0.1))
        self.b1 = tf.Variable(shape=[100], dtype=tf.float32, 
                             initial_value=tf.zeros([100]))
        
        self.W2 = tf.Variable(shape=[100, 10], dtype=tf.float32,
                             initial_value=tf.random.normal([100, 10], stddev=0.1))
        self.b2 = tf.Variable(shape=[10], dtype=tf.float32,
                             initial_value=tf.zeros([10]))
        ####################
        
    def __call__(self, x):
        ####################
        '''实现模型函数体,返回未归一化的logits'''
        # 将输入reshape为 (batch_size, 784)
        x = tf.reshape(x, [-1, 28*28])
        
        # 第一层: x @ W1 + b1, 然后使用ReLU激活
        h1 = tf.matmul(x, self.W1) + self.b1
        h1 = tf.nn.relu(h1)
        
        # 第二层: h1 @ W2 + b2, 输出logits (不使用softmax)
        logits = tf.matmul(h1, self.W2) + self.b2
        ####################
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [9]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    
    # 使用Adam优化器更新参数 (而不是手动梯度下降)
    optimizer.apply_gradients(zip(grads, trainable_vars))

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [10]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.5426521 ; accuracy 0.08573333
epoch 1 : loss 2.3983667 ; accuracy 0.093
epoch 2 : loss 2.2804983 ; accuracy 0.12501666
epoch 3 : loss 2.1819031 ; accuracy 0.19548333
epoch 4 : loss 2.096395 ; accuracy 0.27783334
epoch 5 : loss 2.019216 ; accuracy 0.33971667
epoch 6 : loss 1.9471358 ; accuracy 0.38765
epoch 7 : loss 1.8779787 ; accuracy 0.43123335
epoch 8 : loss 1.810451 ; accuracy 0.47435
epoch 9 : loss 1.7437911 ; accuracy 0.5161167
epoch 10 : loss 1.6776816 ; accuracy 0.55665
epoch 11 : loss 1.6120914 ; accuracy 0.59243333
epoch 12 : loss 1.5472454 ; accuracy 0.6249833
epoch 13 : loss 1.4834919 ; accuracy 0.65396667
epoch 14 : loss 1.4212297 ; accuracy 0.6789333
epoch 15 : loss 1.3609318 ; accuracy 0.69951665
epoch 16 : loss 1.3029815 ; accuracy 0.7165833
epoch 17 : loss 1.2476617 ; accuracy 0.7306167
epoch 18 : loss 1.1952021 ; accuracy 0.74165
epoch 19 : loss 1.1456149 ; accuracy 0.7513833
epoch 20 : loss 1.0987854 ; accuracy 0.75991666
epoch 21 : loss 1.0545367 ; 